# Movie Recommendation System — EDA & Model Evaluation

Covers: TMDB EDA, content-based training & metrics, SVD collaborative filtering, hybrid demo.

In [ ]:
import sys
sys.path.insert(0, '..')
import ast, numpy as np, pandas as pd
import plotly.express as px
from collections import Counter
pd.set_option('display.max_columns', 20)

## 1. Load TMDB Data

In [ ]:
from src.utils import load_tmdb_data
df = load_tmdb_data('../data/tmdb_5000_movies.csv', '../data/tmdb_5000_credits.csv')
print(f'Shape: {df.shape}')
df[['id','title','genres','vote_average','vote_count','budget','revenue']].head(3)

In [ ]:
print('Missing values:')
print(df[['title','overview','genres','keywords','cast','crew']].isnull().sum())

## 2. Exploratory Data Analysis

In [ ]:
all_genres = []
for row in df['genres'].dropna():
    try:
        all_genres.extend(g['name'] for g in ast.literal_eval(row))
    except: pass
gdf = pd.DataFrame(Counter(all_genres).most_common(15), columns=['Genre','Count'])
fig = px.bar(gdf, x='Count', y='Genre', orientation='h', title='Top 15 Genres',
             color='Count', color_continuous_scale='Viridis')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=450)
fig.show()

In [ ]:
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
fig = px.histogram(df.dropna(subset=['release_year']), x='release_year', nbins=40,
                   title='Movies by Release Year', color_discrete_sequence=['#636EFA'])
fig.update_layout(bargap=0.05, height=350)
fig.show()

In [ ]:
fin = df[(df['budget']>0) & (df['revenue']>0)].copy()
fin['log_budget']  = np.log10(fin['budget'])
fin['log_revenue'] = np.log10(fin['revenue'])
fig = px.scatter(fin, x='log_budget', y='log_revenue', opacity=0.4, trendline='ols',
                 title='Budget vs Revenue (log10)',
                 labels={'log_budget':'log10(Budget)','log_revenue':'log10(Revenue)'})
fig.update_layout(height=400)
fig.show()

In [ ]:
fig = px.histogram(df[df['vote_count']>=20], x='vote_average', nbins=30,
                   title='Vote Average Distribution (>=20 votes)',
                   color_discrete_sequence=['#EF553B'])
fig.update_layout(bargap=0.05, height=350)
fig.show()

## 3. Content-Based Filtering

In [ ]:
from src.content_based import ContentBasedRecommender
cb = ContentBasedRecommender()
cb.fit(df)
print(f'Movies in model: {len(cb.get_movie_titles())}')

In [ ]:
query = 'The Dark Knight'
recs = cb.recommend(query, n=10)
print(f'Top 10 similar to {repr(query)}:')
recs

In [ ]:
fig = px.bar(recs, x='similarity_score', y='title', orientation='h',
             title=f'Cosine Similarity to {repr(query)}',
             color='similarity_score', color_continuous_scale='Blues')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=400)
fig.show()

In [ ]:
avg_sim = cb.avg_similarity_score(n=10, sample=500)
print(f'Avg cosine similarity (top-10, 500-movie sample): {avg_sim:.4f}')

## 4. Collaborative Filtering (SVD)

In [ ]:
from src.utils import load_ratings, load_ml_movies
ratings_df   = load_ratings('../data/ratings_small.csv')
ml_movies_df = load_ml_movies('../data/ml_movies.csv')
if ratings_df is None:
    print('MovieLens not found. Download: https://grouplens.org/datasets/movielens/latest/')
else:
    print(f'Ratings: {len(ratings_df):,}  Users: {ratings_df.userId.nunique():,}  Movies: {ratings_df.movieId.nunique():,}')
    display(ratings_df.head())

In [ ]:
if ratings_df is not None:
    fig = px.histogram(ratings_df, x='rating', nbins=10,
                       title='MovieLens Rating Distribution', color_discrete_sequence=['#AB63FA'])
    fig.update_layout(bargap=0.1, height=350)
    fig.show()

In [ ]:
if ratings_df is not None:
    from src.collaborative import CollaborativeRecommender
    collab = CollaborativeRecommender(n_factors=100, n_epochs=20)
    collab.fit(ratings_df, ml_movies_df=ml_movies_df)
    print(f'RMSE: {collab.rmse}  MAE: {collab.mae}')

In [ ]:
if ratings_df is not None:
    user_id = collab.get_user_ids()[0]
    print(f'Top 10 predictions for user {user_id}:')
    display(collab.recommend(user_id, n=10))

## 5. Hybrid Recommender

In [ ]:
if ratings_df is not None:
    from src.hybrid import HybridRecommender
    hybrid = HybridRecommender(cb, collab)
    h_recs = hybrid.recommend('Inception', user_id=user_id, n=10)
    print('Hybrid recommendations (50/50 weight):')
    display(h_recs)

In [ ]:
if ratings_df is not None:
    rows = []
    for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
        r = hybrid.recommend('Inception', user_id=user_id, n=3, content_weight=w)
        rows.append({'content_weight': w, 'top_1': r.iloc[0]['title'] if not r.empty else 'N/A'})
    print('Top-1 recommendation vs content weight:')
    display(pd.DataFrame(rows))

## 6. Metrics Summary

In [ ]:
summary = [
    {'Model': 'Content-Based', 'Algorithm': 'CountVectorizer + Cosine Similarity',
     'Dataset': 'TMDB 5000', 'Key Metric': f'Avg similarity = {avg_sim:.4f}'},
    {'Model': 'Collaborative (SVD)', 'Algorithm': 'Matrix Factorization (Surprise)',
     'Dataset': 'MovieLens Small',
     'Key Metric': f'RMSE={collab.rmse} MAE={collab.mae}' if 'collab' in dir() else 'N/A'},
    {'Model': 'Hybrid', 'Algorithm': 'Weighted avg (adjustable)',
     'Dataset': 'TMDB + MovieLens', 'Key Metric': 'See Streamlit slider'},
]
pd.DataFrame(summary)